In [1]:
import polars as pl

In [2]:
path=['/home/mingyu/Recommand-System/项目/OTTO/data/ensembling/submission_rerank_0575.csv','/home/mingyu/Recommand-System/项目/OTTO/data/ensembling/submission_test_all_0522.csv','/home/mingyu/Recommand-System/项目/OTTO/data/ensembling/submission_matrix_factorization_0493.csv']

In [3]:
# 合并结果，并赋予权重
def read_sub(path,weight=1):
    return (
        pl.read_csv(path).with_columns(pl.col('labels').str.split(by=' '))
        .with_columns(pl.lit(weight).alias('vote'))
        .explode('labels')
        .rename({'labels':'aid'})
        .with_columns(pl.col('aid').cast(pl.UInt32))
        .with_columns(pl.col('vote').cast(pl.UInt8))
            )

In [4]:
sub=[read_sub(p) for p in path]
sub[0].head()

session_type,aid,vote
str,u32,u8
"""12899779_clicks""",59625,1
"""12899779_clicks""",1253524,1
"""12899779_clicks""",737445,1
"""12899779_clicks""",438191,1
"""12899779_clicks""",731692,1


In [ ]:
# outer对于匹配的行进行合并，没有匹配的行保留并填充为null
sub=sub[0].join(sub[1],how='outer',on=['session_type','aid']).join(sub[2],houw='outer',on=['session_type','aid'],suffix='_right2')
sub.head()

/tmp/ipykernel_11391/151462355.py:2: DeprecationWarning: use of `how='outer'` should be replaced with `how='full'`.
(Deprecated in version 0.20.29)
  sub=sub[0].join(sub[1],how='outer',on=['session_type','aid']).join(sub[2],houw='outer',on=['session_type','aid'],suffix='_right2')


In [ ]:
sub=(sub
     .fillna(0)
     .with_column((pl.col('vote')+pl.col('vote_right')+pl.col('vote_right2')).alias('vote_sim'))
     .drop(['vote','vote_right','vote_right2'])
     .sort(by='vote_sum')
     .reverse()
     )
sub.head()

In [ ]:
preds=sub.group_by('session_type').agg([pl.col('aid').head(20).alias('labels')])
preds=preds.with_columns(pl.col('labels').apply(lambda lst:' '.join(str(aid) for aid in lst)))

In [ ]:
preds.wirte_csv('ensemble_submission.csv')